# TopoLS Tutorial

### Step 1: Compile a quantum circuit using TopoLS

In [ ]:
# Let's compile 16 qubits GHZ circuit
!python3 prog.py -f ghz_16 -b 20 -zx 1 -dir 1 -l 4 -r 0 -s 2 -t 2 -i 1000 -csv result -sp 0 --backtrack 1
# Compilation results are saved in result/topols/

#### We explain the arguments here.

-b: Maximum block size for circuit slicing. Although a topology-aware partitioning is used, this argument specifies the upper limit on the size of each block.

-zx: ZX optimization level (0: off, 1: on). Enabling this option activates spider fusion.

-dir: Direction optimization level (0: off, 1: on). Enable this option activates placement exploration in MCTS embedding.

-l: Qubit number per row. Logical qubits are initially placed in a rectangular layout, and this value specifies how many logical qubits are placed in each row.

-r: Initial random seed for circuit compilation.

-s: Number of random seed will be tried.

-t: Wall-clock budget for each MCTS call (seconds). The search is anytime: a larger budget only extends the same search.

-i: Number of iterations for MCTS.

-csv: csv result file name, saved in result/topols/.

--backtrack: when a layer cannot be embedded from the best previous-layer state, retry it from up to k of the other seeds' previous-layer states before falling back to coarser strategies (0 = off).

#### If the circuit is too dense:

-sp: For dense circuit we will spread the quantum gates into different rows. This parameter specifies how many gates are placed in each row. In such cases, it is also recommended to reduce -b to 1 or 2.


In [ ]:
# Dense circuit example
!python3 prog.py -f random_500 -b 1 -zx 1 -dir 1 -l 23 -r 0 -s 2 -t 5 -i 1000 -csv result -sp 70
# The baseline method (DASCOT) achieves a z-length of 43, whereas our method reduces it to 19 with the same spatial footprint, representing a significant improvement.

### Step 2: Transform to tqec bgraph format, and visualization

In [ ]:
# Using 2tqec to transpile the TopoLS compilation result to a TQEC readable format, and visualize the TQEC schedule
# -f specifies the filename of the .pkl file to be used for translation, -p specifies whether to visualize the TQEC schedule. The .png file of the pipe diagram is saved in result/visualizatioin/
!python3 2tqec.py -f ghz_16 -p True
# The 500-qubit diagram is too large for the static renderer; export it only
# (add -i True for an interactive HTML view).
!python3 2tqec.py -f random_500

### Interactive view

`2tqec.py -i True` writes a drag/rotate/zoom-able HTML rendering of the pipe diagram.

In [ ]:
!python3 2tqec.py -f ghz_16 -i True     # result/visualization/ghz_16_interactive.html

### Step 3: Simulate our pipe using TQEC
Note: TQEC is still developing methods for simulating S, T, and (in some cases) H gates. 

Therefore, we recommend simulating only circuits composed of CNOT gates.

In [ ]:
# Given the runtime constraints, we only simulate a CNOT here.
!python3 prog.py -f CNOT -b 20 -zx 1 -dir 1 -l 4 -r 0 -s 2 -t 2 -i 1000 -csv result -sp 0
!python3 2tqec.py -f CNOT -p True

# pipe_sim reads result/bgraph/<name>.bgraph (written by 2tqec.py) and simulates it with TQEC + sinter.
# Simulation results are saved in result/simulation/
!python3 -m topols.tools.pipe_sim -f CNOT